In [ ]:
import json

pcg_data_path = "../data/standardized/20250513-PCG_risk_data.json"

pcg_data_with_catalog = json.load(open(pcg_data_path))
len(pcg_data_with_catalog)
pcg_data = []
for item in pcg_data_with_catalog:
    if item["company"] == "PCG":
        pcg_data.append(item)


len(pcg_data)

66

In [2]:
pcg_data[0]

{'company': 'PCG',
 'risk_cat': 'Operational Risk',
 'risk': 'Accounting errors',
 'risk_desc': ['การแจ้งรายละเอียดค่าใช้จ่ายที่เกิดขึ้นในคลังมีความคลาดเคลื่อน ส่งผลให้ค่าใช้จ่ายด้านโลจิสติกส์ (Logistic Cost) ไม่สะท้อนถึงความเป็นจริง และอาจทำให้การคำนวณค่าใช้จ่ายไม่สมเหตุสมผล\n'],
 'rootcause': ['Mistakes in calculations, data entry, or judgment errors during the accounting process: -'],
 'rootcause_desc': [None],
 'process': ['Finance and Accounting: -'],
 'process_desc': [None],
 'risk_level': 1,
 'risk_score': 2,
 'impact_combined': 1,
 'likelihood_combined': 2.0,
 'control_name': ['การตรวจสอบข้อผิดพลาดทางบัญชีแบบเรียลไทม์'],
 'control_desc': ['Recheck ค่าใช้จ่ายที่เกิดขึ้นจริงกับทางบัญชีแจ้ง -ตรวจสอบร่วมกับกับทางบัญชีและทำการแก้ไข"'],
 'control_rootcause': ['Mistakes in calculations, data entry, or judgment errors during the accounting process: -'],
 'control_process': ['Finance and Accounting: -'],
 'control_design_score': [3],
 'control_effective_score': [3],
 'risk_id': 'OP041',

In [6]:
from pydantic import BaseModel, Field
from typing import List, Optional


class Process(BaseModel):
    """Schema for process data."""

    id: str = Field(..., description="Unique identifier for the process")
    name: str = Field(..., description="Name of the process")
    description: str = Field(..., description="Description of the process")


class RootCause(BaseModel):
    """Schema for root cause data."""

    id: str = Field(..., description="Unique identifier for the root cause")
    name: str = Field(..., description="Name of the root cause")
    description: str = Field(..., description="Description of the root cause")


class RiskScore(BaseModel):
    """Schema for risk scoring data."""

    impact: int = Field(..., ge=1, le=5, description="Impact score (1 to 5)")
    likelihood: int = Field(..., ge=1, le=5, description="Likelihood score (1 to 5)")
    score: int = Field(..., ge=1, le=25, description="Overall risk score (1 to 25)")
    risk_level: int = Field(..., ge=1, le=4, description="Risk level (1, 2, 3, 4)")


class ExistingRisk(BaseModel):
    """Schema for existing risk data."""

    risk_id: str = Field(..., description="Unique identifier for the risk")
    user_id: str = Field(
        ...,
        description="Unique identifier for the user",
    )
    company_id: str = Field(
        ...,
        description="Unique identifier for the company",
    )
    country: str = Field(
        ...,
        description="Unique identifier for the country",
    )

    risk_name: str = Field(..., description="Name of the risk")
    risk_description: str = Field(..., description="Description of the risk")
    processes: List[Process] = Field(
        ..., description="List of processes associated with the risk"
    )
    root_causes: List[RootCause] = Field(
        ..., description="List of root causes for the risk"
    )
    risk_category: str = Field(..., description="Category of the risk")
    score: RiskScore = Field(..., description="Risk scoring information")

In [3]:
example_data = {
    "id": "8a3dcc6d-bcec-4ce7-aec5-dfa38930d9ce",
    "data_level": "user",
    "year_quarter": "2025-Q3",
    "existing_risks": [
        {
            "risk_id": "539",
            "user_id": "d95a656c-8071-707d-3b15-5b955769b84e",
            "company_id": "353",
            "country": "thailand",
            "risk_name": "Non-compliance - business related regulations",
            "risk_description": "Non-compliance with laws or standards pertaining business operation, e.g.,\n- lack of permits and certifications for operations e.g. industrial factories operation or construction/ livestock or fishery businesses\n- non-compliance with standards regarding health and safety for livestock",
            "processes": [{"id": "1", "name": "Human Resources", "description": ""}],
            "root_causes": [
                {
                    "id": "2903",
                    "name": "root_cause_2_EC002_(strategy)_lack_of_strategy_or_policy,_or_inadequate_consideration_in_strategic_planning",
                    "description": "",
                }
            ],
            "risk_category": "Operational Risk",
            "score": {"impact": 3, "likelihood": 3, "score": 9, "risk_level": 2},
            "existing_controls": [],
            "mitigation_plans": [],
        },
        {
            "risk_id": "795",
            "user_id": "d95a656c-8071-707d-3b15-5b955769b84e",
            "company_id": "353",
            "country": "thailand",
            "risk_name": "Ineffective OT Operations Management",
            "risk_description": "This risk refers to situation where the activities related to management of OT operations are not performed properly resulting in system unavailability, slow down performance, or business disruptions etc. Examples of OT operations management include:\n- Capacity management\n- Data backup\n- Access log management\n- OT Change management\n- OT Patch management\n- System acquisition and development\n- OT Incident and problem management\n- OT Business continuity plan",
            "processes": [
                {
                    "id": "163",
                    "name": "Sensor (e.g. Flow meters, , Vibration, Termperature, Pressure)",
                    "description": "",
                }
            ],
            "root_causes": [
                {
                    "id": "4802",
                    "name": "root_cause_2_OG003_lack_of_ot_resource_utilization_monitoring",
                    "description": "",
                }
            ],
            "risk_category": "Operational Risk",
            "score": {"impact": 4, "likelihood": 4, "score": 16, "risk_level": 3},
            "existing_controls": [],
            "mitigation_plans": [],
        },
    ],
}

In [11]:
from typing import List, Optional, Dict, Any
from uuid import uuid4
from pydantic import BaseModel, Field


def to_existing_risk(
    record: Dict[str, Any],
    *,
    user_id: str,
    company_id: str,
    country: str,
) -> Dict[str, Any]:
    """
    Convert a flat record dict to ExistingRisk.
    - Generates random ids for Process.id and RootCause.id.
    - Parses 'Name: Description' patterns when present.
    - Falls back to catalog description if risk_desc is empty.
    """

    def first_text(value: Any) -> str:
        if isinstance(value, list):
            for v in value:
                if isinstance(v, str) and v.strip():
                    return v.strip()
            return ""
        return (value or "").strip()

    def parse_name_desc(
        text: Optional[str], explicit_desc: Optional[str]
    ) -> (str, str):
        text = (text or "").strip()
        name, desc = text, ""
        if ":" in text:
            name_part, desc_part = text.split(":", 1)
            name = name_part.strip()
            desc = desc_part.strip().lstrip("-").strip()
        if explicit_desc and not desc:
            desc = explicit_desc.strip()
        return name or "", desc or ""

    # Risk core fields
    risk_id = str(record.get("risk_id") or uuid4().hex)
    risk_name = first_text(record.get("risk"))
    risk_desc_text = first_text(record.get("risk_desc"))
    risk_desc_catalog = first_text(record.get("risk_desc_catalog"))
    risk_description = risk_desc_text or risk_desc_catalog

    # Score
    impact = int(record.get("impact_combined") or 0)
    likelihood_val = record.get("likelihood_combined") or 0
    likelihood = int(round(float(likelihood_val))) if likelihood_val is not None else 0
    score = int(record.get("risk_score") or (impact * likelihood))
    risk_level = int(record.get("risk_level") or 1)

    score_obj = RiskScore(
        impact=impact,
        likelihood=likelihood,
        score=score,
        risk_level=risk_level,
    )

    # Processes
    process_list: List[str] = list(record.get("process") or [])
    process_desc_list: List[Optional[str]] = list(record.get("process_desc") or [])
    processes: List[Process] = []
    for idx, p_text in enumerate(process_list):
        explicit_desc = process_desc_list[idx] if idx < len(process_desc_list) else None
        name, desc = parse_name_desc(p_text, explicit_desc)
        processes.append(
            Process(
                id=uuid4().hex,
                name=name or "Unknown Process",
                description=desc or "",
            )
        )

    # Root causes
    rootcause_list: List[str] = list(record.get("rootcause") or [])
    rootcause_desc_list: List[Optional[str]] = list(record.get("rootcause_desc") or [])
    root_causes: List[RootCause] = []
    for idx, rc_text in enumerate(rootcause_list):
        explicit_desc = (
            rootcause_desc_list[idx] if idx < len(rootcause_desc_list) else None
        )
        name, desc = parse_name_desc(rc_text, explicit_desc)
        root_causes.append(
            RootCause(
                id=uuid4().hex,
                name=name or "Unknown Root Cause",
                description=desc or "",
            )
        )
    existing_risk = ExistingRisk(
        risk_id=risk_id,
        user_id=user_id,
        company_id=company_id,
        country=country,
        risk_name=risk_name or "Unknown Risk",
        risk_description=risk_description or "",
        processes=processes,
        root_causes=root_causes,
        risk_category=str(record.get("risk_cat") or "Unknown"),
        score=score_obj,
    )
    return existing_risk.model_dump()

In [13]:
pcg_data_existing_risk = to_existing_risk(
    pcg_data[0], user_id="test", company_id="PCG_test", country="Thailand"
)
pcg_data_existing_risk

{'risk_id': 'OP041',
 'user_id': 'test',
 'company_id': 'PCG_test',
 'country': 'Thailand',
 'risk_name': 'Accounting errors',
 'risk_description': 'การแจ้งรายละเอียดค่าใช้จ่ายที่เกิดขึ้นในคลังมีความคลาดเคลื่อน ส่งผลให้ค่าใช้จ่ายด้านโลจิสติกส์ (Logistic Cost) ไม่สะท้อนถึงความเป็นจริง และอาจทำให้การคำนวณค่าใช้จ่ายไม่สมเหตุสมผล',
 'processes': [{'id': 'ef587165f27846119284e0cba9914aca',
   'name': 'Finance and Accounting',
   'description': ''}],
 'root_causes': [{'id': '875f52e4183a41fb85de02f74df80678',
   'name': 'Mistakes in calculations, data entry, or judgment errors during the accounting process',
   'description': ''}],
 'risk_category': 'Operational Risk',
 'score': {'impact': 1, 'likelihood': 2, 'score': 2, 'risk_level': 1}}

In [14]:
pcg_data_existing_risk_list = []
for each_pcg_data in pcg_data:
    pcg_data_existing_risk = to_existing_risk(
        each_pcg_data, user_id="test", company_id="PCG_test", country="Thailand"
    )
    pcg_data_existing_risk_list.append(pcg_data_existing_risk)

In [16]:
# create request json from pcg data
example_pcg_request_dict = {
    "id": uuid4().hex,
    "data_level": "user",
    "year_quarter": "2025-Q1",
    "existing_risks": pcg_data_existing_risk_list,
}
# save to json
with open("example_pcg_request.json", "w") as f:
    json.dump(example_pcg_request_dict, f, indent=4, ensure_ascii=False)